In [ ]:
from google import genai
import os
import glob
import pandas as pd
import logging
import re


#load api key from environment variable
#load dotenv
from dotenv import load_dotenv
load_dotenv()


# Configure logging
o_logger = logging.getLogger(__name__)
o_logger.setLevel(logging.INFO)
file_handler = logging.FileHandler('pipeline.log')
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
o_logger.addHandler(file_handler)


def load_local_dataset(data_path: str) -> pd.DataFrame:
    """
    Robust CSV loader for multilingual news files with:
      - UTF-8 BOM stripping
      - delimiter sniffing (handles commas, tabs, semicolons)
      - tolerant parsing for embedded newlines/quotes
      - disabling pandas' NA coercion so 'NaN'/'NA' stay as text
      - header & cell normalization (trim, NBSP removal)
      - drop only truly empty headline/content; fill missing category
    """
    try:
        # 1) Tolerant read; many Hindi/Bangla files have BOM + embedded newlines.
        try:
            df = pd.read_csv(
                data_path,
                encoding="utf-8-sig",   # strips BOM if present
                sep=None,               # auto-scan delimiter
                engine="python",        # robust with mixed quoting/newlines
                dtype=str,              # keep everything as text
                quotechar='"',
                doublequote=True,
                escapechar="\\",
                keep_default_na=False,  # do NOT treat 'NaN', 'NA', etc. as null
                na_filter=False,        # do NOT auto-convert empties to NaN
                on_bad_lines="skip"     # skip malformed lines instead of crashing
            )
        except UnicodeDecodeError:
            # Some files are plain UTF-8 without BOM
            df = pd.read_csv(
                data_path,
                encoding="utf-8",
                sep=None,
                engine="python",
                dtype=str,
                quotechar='"',
                doublequote=True,
                escapechar="\\",
                keep_default_na=False,
                na_filter=False,
                on_bad_lines="skip"
            )

        # 2) Normalize headers: strip BOM/NBSP/whitespace → lowercase
        df.columns = (
            df.columns.astype(str)
              .str.replace("\ufeff", "", regex=False)  # BOM
              .str.replace("\xa0", " ", regex=False)   # NBSP
              .str.strip()
              .str.lower()
        )

        # 3) Map common aliases to required names
        alias_map = {
            "headline": {"headline", "title", "head", "news_headline"},
            "content":  {"content", "article", "text", "story", "body"},
            "category": {"category", "topic", "section", "label", "class"}
        }
        rename = {}
        for target, candidates in alias_map.items():
            match = next((c for c in df.columns if c in candidates), None)
            if not match:
                raise KeyError(
                    f"Expected a column for '{target}' not found. "
                    f"Available columns: {list(df.columns)}"
                )
            rename[match] = target
        df = df.rename(columns=rename)[["headline", "content", "category"]]

        # 4) Clean cell text: collapse whitespace, remove NBSP
        def _clean(s: str) -> str:
            s = (s or "").replace("\xa0", " ")
            s = re.sub(r"\s+", " ", s, flags=re.UNICODE).strip()
            return s

        for col in ["headline", "content", "category"]:
            df[col] = df[col].astype(str).map(_clean)

        # 5) Drop only truly empty headline/content; fill empty category
        before = len(df)
        mask_empty = (df["headline"] == "") | (df["content"] == "")
        df = df[~mask_empty].copy()
        df.loc[df["category"] == "", "category"] = "misc"
        dropped = before - len(df)

        # 6) Optional: canonicalize category for consistency
        df["category"] = df["category"].str.replace(r"\s+", " ", regex=True).str.strip()

        o_logger.info(
            f"Loaded and normalized dataset '{data_path}' "
            f"→ shape={df.shape}, dropped_empty_rows={dropped}"
        )
        return df

    except Exception as e:
        o_logger.error(f"Error loading dataset from {data_path}: {e}")
        raise


class PromptGenerator:
    @staticmethod
    def zero_shot(title: str, language: str) -> str:
        try:
            if not title or not language:
                raise ValueError("Title and language must be non-empty for zero-shot prompt.")
            prompt = f"Write a news article in {language} about the headline: \"{title}\" return only the article on {language} script only."
            o_logger.debug(f"Zero-shot prompt generated: {prompt}")
            return prompt
        except Exception as e:
            o_logger.error(f"Error in zero_shot prompt generation: {e}")
            raise

    @staticmethod
    def few_shot(title: str, df: pd.DataFrame, category: str, language: str) -> str:
        try:
            if not title or not language or not category:
                raise ValueError("Title, language, and category must be non-empty for few-shot prompt.")
            examples = df[df['category'] == category]
            if examples.empty:
                raise ValueError(f"No examples found for category '{category}' in few-shot prompting.")
            samples = examples.sample(min(2, len(examples)))
            prompt = f"Write a news article in {language} about the headline: \"{title}\" return only the article on {language} script only. "
            prompt += f"Below are {len(samples)} {language} news headline–article pairs from category '{category}':\n"
            for _, row in samples.iterrows():
                h, c = row['headline'], row['content']
                if pd.isna(h) or pd.isna(c):
                    raise ValueError("Example headline or content is null.")
                prompt += f"Headline: \"{h}\"\nArticle: \"{c}\"\n\n"
            o_logger.debug(f"Few-shot prompt generated: {prompt[:60]}...")
            return prompt
        except Exception as e:
            o_logger.error(f"Error in few_shot prompt generation: {e}")
            raise

    @staticmethod
    def chain_of_thought(title: str, language: str) -> tuple[str, str]:
        try:
            if not title or not language:
                raise ValueError("Title and language must be non-empty for chain-of-thought prompt.")
            outline_prompt = (
                f"List the key points and logical structure needed to write a factual"
                f"{language} news article about the headline: \"{title}\"."
            )
            article_template = (
                f"Using the outline below, write a polished {language} news article:\n"  
                f"{{outline}}\n\nWrite the full article."
            )
            o_logger.debug(f"CoT outline prompt generated: {outline_prompt}")
            return outline_prompt, article_template
        except Exception as e:
            o_logger.error(f"Error in chain_of_thought prompt generation: {e}")
            raise

class TogetherLLM:
    def __init__(self, api_key: str, model: str = 'gemini-2.0-flash'):
        try:
            self.client = genai.Client(api_key=api_key)
            self.model = model
            o_logger.info(f"Initialized LLM client with model {model}.")
        except Exception as e:
            o_logger.error(f"Error initializing LLM client: {e}")
            raise

    def generate(self, prompt: str, n: int = 1) -> list[str]:
        try:
            if not prompt:
                raise ValueError("Prompt must be non-empty for generation.")
            texts = []
            for _ in range(n):
                response = self.client.models.generate_content(
                    model=self.model,
                    contents=prompt,
                )
                text = response.text.strip()
                if not text:
                    raise ValueError("Generated text is empty.")
                texts.append(text)
            o_logger.info(f"Received {len(texts)} outputs.")
            return texts
        except Exception as e:
            o_logger.error(f"Error during LLM generation: {e}")
            raise

class NewsDatasetGenerator:
    def __init__(
        self,
        llm: TogetherLLM,
        dataset_paths: dict,
        models: list[str],
        samples_per_language: int = 3,
        output_csv: str = 'generated_dataset.csv'
    ):
        self.llm = llm
        self.dataset_paths = dataset_paths
        self.models = models
        self.samples_per_language = samples_per_language
        self.output_csv = output_csv
        self.counter = {}
        os.makedirs('articles', exist_ok=True)
        o_logger.info("Initialized NewsDatasetGenerator with multilingual support.")

    def prepare_data(self) -> pd.DataFrame:
        dfs = []
        for lang, path in self.dataset_paths.items():
            try:
                df = load_local_dataset(path)
                df['language'] = lang
                dfs.append(df)
            except Exception:
                continue
        if not dfs:
            raise RuntimeError("No datasets loaded; please check dataset paths.")
        combined = pd.concat(dfs, ignore_index=True)
        return combined

    def _get_filename(self, language: str, technique: str, category: str, model_name: str) -> str:
        self.counter[(language, technique, category, model_name)] = self.counter.get((language, technique, category, model_name), 0) + 1
        num = self.counter[(language, technique, category, model_name)]
        safe_cat = category.replace(' ', '_')
        safe_model = model_name.replace('.', '_')
        return os.path.join('articles', f"{language}_{safe_model}_{technique}_{safe_cat}_{num}.txt")

    def generate(self) -> None:
        df = self.prepare_data()
        records = []
        for language in df['language'].unique():
            df_lang = df[df['language'] == language]
            total = self.samples_per_language
            props = df_lang['category'].value_counts(normalize=True)
            print(f"Generating {total} samples for language '{language}' with category distribution:\n{props}")
            cats = list(props.index)
            counts = [int(props[cat] * total) for cat in cats[:-1]]

            counts.append(total - sum(counts))
            print(f"Adjusted counts for categories: {dict(zip(cats, counts))}")
            for category, n_samples in zip(cats, counts):
                if n_samples <= 0:
                    continue
                df_cat = df_lang[df_lang['category'] == category]
                replace = len(df_cat) < n_samples
                titles = df_cat['headline'].sample(n=n_samples, replace=replace)
                print(f"Generating {len(titles)} samples for category '{category}' in language '{language}'")
                for title in titles:
                    for technique in ['zero-shot', 'few-shot', 'chain-of-thought']:
                        try:
                            if technique == 'zero-shot':
                                prompt = PromptGenerator.zero_shot(title, language)
                            elif technique == 'few-shot':
                                prompt = PromptGenerator.few_shot(title, df_lang, category, language)
                            else:
                                outline_prompt, template = PromptGenerator.chain_of_thought(title, language)
                                outline = self.llm.generate(outline_prompt, 1)[0]
                                prompt = template.replace('{outline}', outline)
                            for model_name in self.models:
                                self.llm.model = model_name
                                article = self.llm.generate(prompt, 1)[0]
                                fname = self._get_filename(language, technique, category, model_name)
                                with open(fname, 'w', encoding='utf-8') as f:
                                    f.write(article)
                                records.append({
                                    'language': language,
                                    'category': category,
                                    'headline': title,
                                    'prompting_technique': technique,
                                    'model': model_name,
                                    'article_file': fname
                                })
                        except Exception as e:
                            o_logger.error(f"Skipping generation for title '{title}' ({technique}/{model_name}): {e}")
        pd.DataFrame(records).to_csv(self.output_csv, index=False)
        #save to JSON file 
        pd.DataFrame(records).to_json(self.output_csv.replace('.csv', '.json'), orient='records', force_ascii=False, indent=4)
        o_logger.info(f"Generated dataset saved to {self.output_csv}")

if __name__ == '__main__':
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY environment variable not set. Please set it to your Gemini API key.")
    # Set up the API client
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY environment variable not set. Please set it to your OpenAI API key.")
    dataset_paths = {
        'hindi': 'data/bbc_hindi_articles_with_categories_cleaned.csv',
        # 'marathi': 'data/marathi',
        # 'gujarati': 'data/gujarati',
        'tamil': 'data/tamilmurasu_dataset.csv',
        'telugu': 'data/telugu_news_test.csv',
        # 'malayalam': 'data/malayalam',
        'bangla': 'data/bangla_newspaper_dataset.csv'
    }
    models = ['gemini-2.5-pro', 'gemini-2.5-flash']
    llm_client = TogetherLLM(api_key=GEMINI_API_KEY)
    generator = NewsDatasetGenerator(llm_client, dataset_paths, models, samples_per_language=100, output_csv='generated_dataset.csv')
    generator.generate()


Generating 10 samples for language 'tamil' with category distribution:
category
தமிழகம்              0.420786
இந்தியா              0.133614
குற்றம்              0.128525
சினிமா(ரீல்மா)       0.072965
மாவட்ட மசாலா         0.071631
விளையாட்டு           0.064933
உலகம்                0.058992
ஸ்டேட் எக்ஸ்பிரஸ்    0.017776
தலையங்கம்            0.012111
வேலைவாய்ப்பு         0.008221
மருத்துவம்           0.004292
ஆன்மீகம்             0.003203
கல்வி                0.001894
தொழில்               0.000537
மர்மம்               0.000521
Name: proportion, dtype: float64
Adjusted counts for categories: {'தமிழகம்': 4, 'இந்தியா': 1, 'குற்றம்': 1, 'சினிமா(ரீல்மா)': 0, 'மாவட்ட மசாலா': 0, 'விளையாட்டு': 0, 'உலகம்': 0, 'ஸ்டேட் எக்ஸ்பிரஸ்': 0, 'தலையங்கம்': 0, 'வேலைவாய்ப்பு': 0, 'மருத்துவம்': 0, 'ஆன்மீகம்': 0, 'கல்வி': 0, 'தொழில்': 0, 'மர்மம்': 4}
Generating 4 samples for category 'தமிழகம்' in language 'tamil'


KeyboardInterrupt: 

In [2]:
!pip install google-genai


In [ ]:
import pandas as pd
df = pd.read_parquet('data/telugu_news_test.parquet')
print(df.head())

#save the dataset to a CSV file
df.to_csv('telugu_news_test.csv', index=False)
print("Dataset saved to telugu_news_test.csv")


                                                   title  \
18434                             జీవన్మృతురాలికి పాపాయి   
7379                       అంతర్‌ జిల్లా ఫుట్‌బాల్‌ షురూ   
784                                         పంచ్‌డైలాగ్‌   
13645  వచ్చే ఏడాదిలో కొత్త క్షేత్రాల్లో గ్యాస్‌ ఉత్పత్తి   
14701      23 నెలల కనిష్ఠానికి.. టోకు ద్రవ్యోల్బణం 2.02%   

                                                    text         category  \
18434  \n                ప్రాగ్‌: జీవన్మృతురాలై(బ్రెయ...  eenadu_national   
7379   \n                ఈనాడు డిజిటల్‌, హైదరాబాద్‌: ...    eenadu_sports   
784    \n                \n\n\nబంధం కోసం పంతాన్ని వదు...    eenadu_cinema   
13645  \n                దిల్లీ: ప్రతిష్ఠాత్మక కేజీ- ...  eenadu_business   
14701  \n                \n\nదిల్లీ: కూరగాయలు, ఇంధనం,...  eenadu_business   

                                                       t  
18434  జీవన్మృతురాలికి పాపాయి \n                ప్రాగ...  
7379   అంతర్‌ జిల్లా ఫుట్‌బాల్‌ షురూ \n              ...  

KeyError: 0

In [3]:
from datasets import load_dataset

ds = load_dataset("zabir-nabil/bangla_newspaper_dataset")



/Users/suyashsethia/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/suyashsethia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test_2 split: 100%|██████████| 495/495 [00:00<00:00, 99615.22 examples/s]


In [4]:
# save the dataset to a CSV file
ds['test_1'].to_csv('bangla_newspaper_dataset.csv', index=False)


Creating CSV from Arrow format: 100%|██████████| 82/82 [00:02<00:00, 30.45ba/s]


423482294

In [5]:
with open("data/DataSet/114010164688.utf8", "r", encoding="utf-8") as f:
        content = f.read()
        print(content)

<DOC>
<DOCNO>114010164688.utf8</DOCNO>
<TEXT>


            ഗോതുരുത്ത് ഉത്സവ് ഇന്ന് കാര്‍ണിവല്‍    

Wednesday 1 January 2014 12:41 am IST

        പറവൂര്‍:  പുതുവത്സര വേളയില്‍ നാടിനു പുതുവത്സര സമ്മാനമായി ഗോതുരുത്ത്  മുസിരീസ്  ഗ്രാമത്തില്‍ ഇന്ന് വൈകിട്ട് 3 മണിക്ക് കാര്‍ണിവല്‍ നടക്കും. കൊച്ചി  കാര്‍ണിവലിനു സമാനമായി എറണാകുളം ജില്ലയിലെ കാര്‍ണിവലായി അറിയപ്പെടുന്ന  ഗോതുരുത്ത് കാര്‍ണിവലില്‍ വിഷയാവതരണത്തോടെയുള്ള ഫ്‌ളോട്ടുകള്‍,  ഫാന്‍സിഡ്രസ് വേഷധാരികള്‍, വിവിധ കലാരൂപങ്ങള്‍ എന്നിവ വാദ്യമേളങ്ങളുടെ  അകമ്പടിയോടെ വര്‍ണശബളമായ ഘോഷയാത്രയില്‍ പങ്കുചേരും. മുന്‍ വര്‍ഷത്തേക്കാള്‍  കൂടുതല്‍ വിദേശികളും, അന്യ സംസ്ഥാന കലാകാരന്മാരും എത്തുമെനന്നുള്ള  പ്രതീക്ഷയിലാണ് സംഘാടകര്‍, രാവിലെ 9 മണിയോടെ പുതുവത്സര പരിപാടികള്‍ക്ക്  തുടക്കമാകും. ഗ്രാമത്തിലേയും സമീപ പ്രദേശങ്ങളിലേയും വിവിധ കലാസംഘങ്ങളുടെ  മികച്ച പ്രകടനത്തോടെയാണ് ലൈവ് സ്റ്റേജ്‌ഷോ തുടങ്ങുന്നത്. ഗ്രാമത്തിന്റെ  തനത് രുചിക്കുട്ടുകള്‍ മനസ്സിലാക്കുന്നതിനും രുചിച്ചറിയുന്നതിനും  ഗ്രാമവാസികള്‍ ഒരുക്കുന്ന 20 ഓളം സ്റ്റാളുകളും നാടന്‍ ഭക്ഷ്യമേളയില്‍  ഒരുക്കുന്നുണ്ട്. തൃപ്പൂണ

In [6]:
#open data/bbc_hindi_articles_with_categories_cleaned.csv

import pandas as pd
df = pd.read_csv('data/bbc_hindi_articles_with_categories_cleaned.csv')
print(df.head())

                                            Headline  \
0  बांग्लादेश जमात-ए-इस्लामी और मोदी-बाइडन की बात...   
1        भारत में महिलाएं क्यों छिपाती हैं मेनोपॉज़?   
2  वीडियो, 'वडोदरा शहर में करोड़ों रुपए के बंगले ...   
3  अवनि लेखरा: टोक्यो के बाद पेरिस में भी गोल्ड प...   
4  पैरालंपिक में भाग्यश्री जाधव: मेहनत और लगन से ...   

                                             Content Category  
0  प्रधानमंत्री नरेंद्र मोदी और अमेरिका राष्ट्रपत...     भारत  
1  “मेरे पति चाहते थे कि मैं हमेशा तैयार और सज सं...     भारत  
2  गुजरात में लगातार हो रही भारी बारिश के कारण कई...     भारत  
3  अवनि लेखरा ने पेरिस पैरालंपिक में महिलाओं की 1...     भारत  
4  पेरिस पैरालंपिक खेलों की शुरुआत 28 अगस्त से हो...     भारत  


In [3]:
! pip install together

  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.6 MB/s eta 0:00:009 MB/s eta 0:00:01
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 7.0 MB/s eta 0:00:00 MB/s eta 0:00:01:01
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)


In [ ]:
# -*- coding: utf-8 -*-
from itertools import count

from __future__ import annotations

import os
import re
import math
import json
import time
import random
import logging
import threading
from typing import Dict, List, Tuple, Optional
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

# === External SDKs ===
# Keep your existing import for Google GenAI (newer "google-genai" SDK)
from google import genai  # pip install google-genai
# OpenAI SDK (>=1.0)
from openai import OpenAI  # pip install openai
# Together SDK
from together import Together  # pip install together

# === .env ===
from dotenv import load_dotenv
load_dotenv()

# === Logging ===
o_logger = logging.getLogger(__name__)
o_logger.setLevel(logging.INFO)
if not o_logger.handlers:
    fh = logging.FileHandler('pipeline.log', encoding='utf-8')
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(fmt)
    o_logger.addHandler(fh)

# =========================
# Dataset loader (unchanged)
# =========================
def load_local_dataset(data_path: str) -> pd.DataFrame:
    """
    Robust CSV loader for multilingual news files with:
      - UTF-8 BOM stripping
      - delimiter sniffing (handles commas, tabs, semicolons)
      - tolerant parsing for embedded newlines/quotes
      - disabling pandas' NA coercion so 'NaN'/'NA' stay as text
      - header & cell normalization (trim, NBSP removal)
      - drop only truly empty headline/content; fill missing category
    """
    try:
        # 1) Tolerant read; many Hindi/Bangla files have BOM + embedded newlines.
        try:
            df = pd.read_csv(
                data_path,
                encoding="utf-8-sig",   # strips BOM if present
                sep=None,               # auto-scan delimiter
                engine="python",        # robust with mixed quoting/newlines
                dtype=str,              # keep everything as text
                quotechar='"',
                doublequote=True,
                escapechar="\\",
                keep_default_na=False,  # do NOT treat 'NaN', 'NA', etc. as null
                na_filter=False,        # do NOT auto-convert empties to NaN
                on_bad_lines="skip"     # skip malformed lines instead of crashing
            )
        except UnicodeDecodeError:
            # Some files are plain UTF-8 without BOM
            df = pd.read_csv(
                data_path,
                encoding="utf-8",
                sep=None,
                engine="python",
                dtype=str,
                quotechar='"',
                doublequote=True,
                escapechar="\\",
                keep_default_na=False,
                na_filter=False,
                on_bad_lines="skip"
            )

        # 2) Normalize headers: strip BOM/NBSP/whitespace → lowercase
        df.columns = (
            df.columns.astype(str)
              .str.replace("\ufeff", "", regex=False)  # BOM
              .str.replace("\xa0", " ", regex=False)   # NBSP
              .str.strip()
              .str.lower()
        )

        # 3) Map common aliases to required names
        alias_map = {
            "headline": {"headline", "title", "head", "news_headline"},
            "content":  {"content", "article", "text", "story", "body"},
            "category": {"category", "topic", "section", "label", "class"}
        }
        rename = {}
        for target, candidates in alias_map.items():
            match = next((c for c in df.columns if c in candidates), None)
            if not match:
                raise KeyError(
                    f"Expected a column for '{target}' not found. "
                    f"Available columns: {list(df.columns)}"
                )
            rename[match] = target
        df = df.rename(columns=rename)[["headline", "content", "category"]]

        # 4) Clean cell text: collapse whitespace, remove NBSP
        def _clean(s: str) -> str:
            s = (s or "").replace("\xa0", " ")
            s = re.sub(r"\s+", " ", s, flags=re.UNICODE).strip()
            return s

        for col in ["headline", "content", "category"]:
            df[col] = df[col].astype(str).map(_clean)

        # 5) Drop only truly empty headline/content; fill empty category
        before = len(df)
        mask_empty = (df["headline"] == "") | (df["content"] == "")
        df = df[~mask_empty].copy()
        df.loc[df["category"] == "", "category"] = "misc"
        dropped = before - len(df)

        # 6) Optional: canonicalize category for consistency
        df["category"] = df["category"].str.replace(r"\s+", " ", regex=True).str.strip()

        o_logger.info(
            f"Loaded and normalized dataset '{data_path}' "
            f"→ shape={df.shape}, dropped_empty_rows={dropped}"
        )
        return df

    except Exception as e:
        o_logger.error(f"Error loading dataset from {data_path}: {e}")
        raise
# =========================
# Prompt generators (kept)
# =========================
class PromptGenerator:
    @staticmethod
    def zero_shot(title: str, language: str) -> str:
        if not title or not language:
            raise ValueError("Title and language must be non-empty for zero-shot prompt.")
        return (
            f"Write a news article in {language} about the headline: \"{title}\". "
            f"Return only the article in {language} script."
        )

    @staticmethod
    def few_shot(title: str, df: pd.DataFrame, category: str, language: str) -> str:
        if not title or not language or not category:
            raise ValueError("Title, language, and category must be non-empty for few-shot prompt.")
        examples = df[df['category'] == category]
        if examples.empty:
            raise ValueError(f"No examples in category '{category}' for few-shot.")
        samples = examples.sample(min(2, len(examples)))
        prompt = (
            f"Write a news article in {language} about the headline: \"{title}\". "
            f"Return only the article in {language} script.\n"
            f"Below are {len(samples)} {language} headline–article pairs from '{category}':\n"
        )
        for _, row in samples.iterrows():
            h, c = row['headline'], row['content']
            prompt += f"Headline: \"{h}\"\nArticle: \"{c}\"\n\n"
        return prompt

    @staticmethod
    def chain_of_thought(title: str, language: str) -> tuple[str, str]:
        if not title or not language:
            raise ValueError("Title and language must be non-empty for chain-of-thought prompt.")
        outline_prompt = (
            f"Draft a concise bullet outline of key points to write a factual {language} news article "
            f"about the headline: \"{title}\". Keep it short and high-level."
        )
        article_template = (
            f"Using the outline below, write a polished {language} news article:\n"
            f"{{outline}}\n\nReturn only the full article in {language} script."
        )
        return outline_prompt, article_template

# ======================================
# Multi-provider LLM Router (NEW)
# ======================================
# --- drop-in replacement for LLMRouter (only this class changed) ---
class LLMRouter:
    """
    Routes generation requests to Gemini, OpenAI, or Together.
    Now supports:
      - 'gemma-3-12b-it' via Gemini API
      - 'mistralai/Mistral-7B-Instruct-v0.2' via Together API
    Thread-safe with per-provider locks. Includes simple retries.
    """
    def __init__(self, gemini_key: Optional[str], openai_key: Optional[str], together_key: Optional[str]):
        self.gemini_key = gemini_key
        self.openai_key = openai_key
        self.together_key = together_key

        # SDK clients
        self._gemini_client = genai.Client(api_key=self.gemini_key) if self.gemini_key else None
        self._openai_client = OpenAI(api_key=self.openai_key) if self.openai_key else None
        self._together_client = Together(api_key=self.together_key) if self.together_key else None

        # Locks per provider
        self._locks = {
            'gemini': threading.Lock(),
            'openai': threading.Lock(),
            'together': threading.Lock(),
        }

        # Friendly aliases -> provider-native model ids (extend as needed)
        self.TOGETHER_MODEL_MAP = {
            "qwen-3": "Qwen/Qwen2.5-72B-Instruct",
            # shorthand alias if you ever pass just "mistral-7b-instruct-v0.2"
            "mistral-7b-instruct-v0.2": "mistralai/Mistral-7B-Instruct-v0.2",
        }

    def generate_text(self, model: str, prompt: str, max_retries: int = 5, backoff: float = 1.5) -> str:
        """
        Routing rules:
          - Gemini API: model startswith 'gemini' or 'gemma'
          - OpenAI API: model startswith 'gpt-'
          - Together API: model contains '/' (e.g., 'mistralai/...') OR startswith any of
            ('qwen', 'mistral', 'meta-llama', 'llama')
          - Default fallback: Gemini
        """
        m = model.lower()

        if m.startswith(("gemini", "gemma")):
            return self._retry(lambda: self._gen_gemini(model, prompt), max_retries, backoff)
        elif m.startswith("gpt-"):
            return self._retry(lambda: self._gen_openai(model, prompt), max_retries, backoff)
        elif ("/" in model) or m.startswith(("qwen", "mistral", "meta-llama", "llama")):
            return self._retry(lambda: self._gen_together_chat(model, prompt), max_retries, backoff)
        else:
            # Conservative fallback to Gemini
            return self._retry(lambda: self._gen_gemini(model, prompt), max_retries, backoff)

    def _retry(self, fn, max_retries, backoff):
        last = None
        for i in range(max_retries):
            try:
                return fn()
            except Exception as e:
                last = e
                time.sleep((backoff ** i) + random.random() * 0.1)
        raise last

    # --- Providers ---

    def _gen_gemini(self, model: str, prompt: str) -> str:
        """
        Google GenAI path. Works for both 'gemini-*' and 'gemma-*' families.
        Example models:
          - 'gemini-2.5-pro', 'gemini-2.5-flash'
          - 'gemma-3-12b-it'
        """
        if not self._gemini_client:
            raise RuntimeError("Gemini client unavailable (missing GEMINI_API_KEY).")
        with self._locks['gemini']:
            resp = self._gemini_client.models.generate_content(model=model, contents=prompt)
        text = (getattr(resp, "text", None) or "").strip()
        if not text:
            raise ValueError("Empty response from Gemini.")
        return text

    def _gen_openai(self, model: str, prompt: str) -> str:
        """OpenAI chat completions path for 'gpt-*' models."""
        if not self._openai_client:
            raise RuntimeError("OpenAI client unavailable (missing OPENAI_API_KEY).")
        with self._locks['openai']:
            resp = self._openai_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
            )
        msg = resp.choices[0].message.content if resp.choices and resp.choices[0].message else ""
        text = (msg or "").strip()
        if not text:
            raise ValueError("Empty response from OpenAI.")
        return text

    def _gen_together_chat(self, model: str, prompt: str) -> str:
        """
        Generic Together chat path. Accepts full repo-style ids like
        'mistralai/Mistral-7B-Instruct-v0.2' and friendly aliases defined in TOGETHER_MODEL_MAP.
        Also used for Qwen routing.
        """
        if not self._together_client:
            raise RuntimeError("Together client unavailable (missing TOGETHER_API_KEY).")
        together_model = self.TOGETHER_MODEL_MAP.get(model.lower(), model)
        with self._locks['together']:
            resp = self._together_client.chat.completions.create(
                model=together_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
            )
        text = resp.choices[0].message.content if resp.choices and resp.choices[0].message else ""
        text = (text or "").strip()
        if not text:
            raise ValueError("Empty response from Together.")
        return text

    # Backward-compatible wrapper name kept for any existing references.
    def _gen_together_qwen(self, model: str, prompt: str) -> str:
        return self._gen_together_chat(model, prompt)


# ======================================
# Utility: distribution helpers (NEW)
# ======================================
def even_split(total: int, k: int) -> List[int]:
    """Split total as evenly as possible across k buckets."""
    base, rem = divmod(total, k)
    return [base + (1 if i < rem else 0) for i in range(k)]

def proportional_counts(total: int, weights: pd.Series) -> Dict[str, int]:
    """Integer counts per key proportional to weights; last key gets remainder."""
    weights = weights[weights > 0]
    if weights.empty:
        return {}
    keys = list(weights.index)
    props = (weights / weights.sum()).to_list()
    raw = [int(total * p) for p in props[:-1]]
    last = total - sum(raw)
    counts = raw + [last]
    return dict(zip(keys, counts))

# ======================================
# News Dataset Generator (updated)
# ======================================
class NewsDatasetGenerator:
    def __init__(
        self,
        llm_router: LLMRouter,
        dataset_paths: Dict[str, str],
        models: List[str],
        output_dir: str = 'articles',
        output_csv: str = 'generated_dataset.csv',
        total_articles: int = 10_000,
        max_workers: int = 10,
        skip_seen_datapoints=True,
        shuffle_seed=None
    ):
        self.llm = llm_router
        self.dataset_paths = dataset_paths
        self.models = models
        self.output_dir = output_dir
        self.output_csv = output_csv
        self.total_articles = int(total_articles)
        self.max_workers = max_workers

        os.makedirs(self.output_dir, exist_ok=True)
        # self._filename_counter = Counter()
        # self._fn_lock = threading.Lock()
        self._uid_lock = threading.Lock()
        self._uid_counter = count(self._bootstrap_uid_start(self.output_dir))
        self._records_lock = threading.Lock()
        self._records: List[Dict] = []
        self.skip_seen_datapoints = bool(skip_seen_datapoints)
        self.shuffle_seed = shuffle_seed

        self.techniques = ['zero-shot', 'few-shot', 'chain-of-thought']

    def _bootstrap_uid_start(self, output_dir: str) -> int:
        """
        Find the next UID to use by scanning existing files in output_dir.
        This makes UIDs unique across reruns as well (not just within one run).
        Returns the first UID to be used (starts at max_seen+1, or 1 if none).
        """
        max_seen = 0
        pat = re.compile(r'_(\d+)\.txt$')
        try:
            if os.path.isdir(output_dir):
                for name in os.listdir(output_dir):
                    m = pat.search(name)
                    if m:
                        max_seen = max(max_seen, int(m.group(1)))
        except Exception as e:
            o_logger.warning(f"UID bootstrap scan failed for '{output_dir}': {e}")
        return max_seen + 1

    def _next_uid(self) -> str:
        """
        Thread-safe unique numeric id (supports up to 20k easily).
        Format is fixed-width for nicer sorting; adjust width if you want.
        """
        with self._uid_lock:
            uid = next(self._uid_counter)
        return f"{uid:05d}"   # 00001 .. 20000

    def prepare_data(self) -> pd.DataFrame:
        dfs = []
        for lang, path in self.dataset_paths.items():
            try:
                df = load_local_dataset(path)
                df['language'] = lang
                dfs.append(df)
            except Exception as e:
                o_logger.error(f"Skipping language {lang}: {e}")
        if not dfs:
            raise RuntimeError("No datasets loaded; please check paths.")
        df = pd.concat(dfs, ignore_index=True)

        # 1) Drop datapoints already used in previous runs
        if self.skip_seen_datapoints:
            seen = self._load_seen_keys()
            if seen:
                df["__key__"] = list(zip(df["language"], df["category"], df["headline"]))
                before = len(df)
                df = df[~df["__key__"].isin(seen)].drop(columns="__key__")
                o_logger.info(f"Filtered {before - len(df)} previously-used datapoints; remaining={len(df)}")

        # 2) Shuffle the remaining pool
        seed = self.shuffle_seed if self.shuffle_seed is not None else int(time.time()) & 0xFFFFFFFF
        df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        return df

    def _safe_filename(self, language: str, technique: str, category: str, model_name: str) -> str:
        safe_cat = re.sub(r'\W+', '_', str(category)).strip('_')
        safe_model = re.sub(r'\W+', '_', str(model_name)).strip('_')

        while True:
            uid = self._next_uid()
            path = os.path.join(self.output_dir, f"{language}_{safe_model}_{technique}_{safe_cat}_{uid}.txt")
            if not os.path.exists(path):
                return path    
    # def _safe_filename(self, language: str, technique: str, category: str, model_name: str) -> str:
    #     safe_cat = re.sub(r'\W+', '_', str(category)).strip('_')
    #     safe_model = re.sub(r'\W+', '_', str(model_name)).strip('_')
    #     key = (language, technique, safe_cat, safe_model)
    #     with self._fn_lock:
    #         self._filename_counter[key] += 1
    #         num = self._filename_counter[key]
    #     return os.path.join(self.output_dir, f"{language}_{safe_model}_{technique}_{safe_cat}_{num}.txt")
    def _load_seen_keys(self) -> set[tuple]:
        """Read previously generated datapoints so we don't reuse them."""
        seen = set()
        try:
            if os.path.exists(self.output_csv):
                prev = pd.read_csv(self.output_csv, dtype=str, keep_default_na=False)
                for _, r in prev.iterrows():
                    seen.add((r.get("language",""), r.get("category",""), r.get("headline","")))
        except Exception as e:
            o_logger.warning(f"Could not load existing outputs for de-dup: {e}")
        return seen
    def _build_plan(self, df: pd.DataFrame) -> List[Dict]:
        """
        Build a task list of exactly self.total_articles meeting:
          - even across languages
          - even across techniques
          - proportional across categories (within language)
          - even across models (within each language+technique)
        """
        tasks: List[Dict] = []
        languages = list(df['language'].unique())
        per_lang = even_split(self.total_articles, len(languages))

        for lang, lang_total in zip(languages, per_lang):
            df_lang = df[df['language'] == lang].copy()
            # Even across techniques
            per_tech = even_split(lang_total, len(self.techniques))
            cat_props = df_lang['category'].value_counts(normalize=True)

            for technique, tech_total in zip(self.techniques, per_tech):
                # Proportional across categories
                cat_counts = proportional_counts(tech_total, cat_props)
                # Even across models for each category
                for category, cat_total in cat_counts.items():
                    if cat_total <= 0:
                        continue
                    df_cat = df_lang[df_lang['category'] == category]
                    titles = df_cat['headline']
                    if titles.empty:
                        continue
                    per_model = even_split(cat_total, len(self.models))
                    # cycle through titles with replacement if needed
                    titles_list = titles.tolist()
                    ti = 0
                    for model, mcount in zip(self.models, per_model):
                        for _ in range(mcount):
                            title = titles_list[ti % len(titles_list)]
                            ti += 1
                            tasks.append({
                                'language': lang,
                                'category': category,
                                'headline': title,
                                'technique': technique,
                                'model': model
                            })
        # In rare rounding cases, adjust to exact total:
        if len(tasks) > self.total_articles:
            tasks = tasks[:self.total_articles]
        elif len(tasks) < self.total_articles:
            # Pad by duplicating earliest tasks (rare)
            need = self.total_articles - len(tasks)
            tasks.extend(tasks[:need])
        random.shuffle(tasks)
        return tasks

    def _make_prompt(self, task: Dict, df_lang: pd.DataFrame) -> Tuple[str, Optional[str]]:
        """Return (prompt, outline_prompt_if_any)"""
        title = task['headline']
        language = task['language']
        category = task['category']
        technique = task['technique']

        if technique == 'zero-shot':
            return PromptGenerator.zero_shot(title, language), None
        elif technique == 'few-shot':
            return PromptGenerator.few_shot(title, df_lang, category, language), None
        else:
            outline_prompt, template = PromptGenerator.chain_of_thought(title, language)
            return template, outline_prompt  # we’ll fill outline before final gen

    def _process_one(self, df: pd.DataFrame, task: Dict) -> Optional[Dict]:
        """Generate one article and write it to disk; return record for CSV."""
        lang = task['language']
        model = task['model']
        cat = task['category']
        tech = task['technique']
        title = task['headline']

        df_lang = df[df['language'] == lang]
        prompt_or_template, outline_prompt = self._make_prompt(task, df_lang)

        try:
            if outline_prompt is not None:
                # CoT outline first (same model)
                outline = self.llm.generate_text(model, outline_prompt)
                prompt = prompt_or_template.replace('{outline}', outline)
            else:
                prompt = prompt_or_template

            article = self.llm.generate_text(model, prompt)
            if not article.strip():
                raise ValueError("Empty article text.")

            fname = self._safe_filename(lang, tech, cat, model)
            with open(fname, 'w', encoding='utf-8') as f:
                f.write(article)
            
            print(f"[GENERATED] {fname}", flush=True)






            record = {
                'language': lang,
                'category': cat,
                'headline': title,
                'prompting_technique': tech,
                'model': model,
                'article_file': fname,
                'article_text': article
            }

            with self._records_lock:
                self._records.append(record)

            # Per-article log entry
            o_logger.info(
                json.dumps({
                    "event": "article_generated",
                    "file": fname,
                    "language": lang,
                    "technique": tech,
                    "category": cat,
                    "model": model,
                    "headline": title
                }, ensure_ascii=False)
            )
            return record

        except Exception as e:
            o_logger.error(f"Generation failed [{lang}/{tech}/{cat}/{model}] '{title}': {e}")
            return None

    def _validate_and_report(self, df_src: pd.DataFrame, out_df: pd.DataFrame) -> None:
        """Log checks for distribution constraints."""
        try:
            total = len(out_df)
            lang_counts = out_df['language'].value_counts().to_dict()
            tech_counts = out_df['prompting_technique'].value_counts().to_dict()

            o_logger.info(f"[CHECK] Total articles: {total} (target={self.total_articles})")
            o_logger.info(f"[CHECK] By language: {lang_counts}")
            o_logger.info(f"[CHECK] By technique: {tech_counts}")

            # Per-language category proportionality (compare KL or simple diff)
            for lang in out_df['language'].unique():
                src_props = df_src[df_src['language']==lang]['category'].value_counts(normalize=True)
                gen_props = out_df[out_df['language']==lang]['category'].value_counts(normalize=True)
                # report L1 distance
                cats = set(src_props.index) | set(gen_props.index)
                l1 = sum(abs(float(src_props.get(c,0))-float(gen_props.get(c,0))) for c in cats)
                o_logger.info(f"[CHECK] L1 distance for category proportions in '{lang}': {l1:.4f}")
        except Exception as e:
            o_logger.error(f"Validation/report error: {e}")

    def generate(self) -> None:
        df = self.prepare_data()
        tasks = self._build_plan(df)

        o_logger.info(f"Planned {len(tasks)} tasks; starting pool with {self.max_workers} workers...")
        with ThreadPoolExecutor(max_workers=self.max_workers) as ex:
            futures = [ex.submit(self._process_one, df, t) for t in tasks]
            for _ in as_completed(futures):
                pass  # progress handled via logging

        out_df = pd.DataFrame(self._records)
        out_df.to_csv(self.output_csv, index=False)
        out_df.to_json(self.output_csv.replace('.csv', '.json'), orient='records', force_ascii=False, indent=2)
        o_logger.info(f"Saved CSV to {self.output_csv} and JSON to {self.output_csv.replace('.csv','.json')}")

        self._validate_and_report(df, out_df)

# =========================
# Main
# =========================
if __name__ == '__main__':
    # Required keys in .env
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    TOGETHER_API_KEY = os.getenv('TOGETHER_API_KEY')

    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY not set.")
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY not set.")
    if not TOGETHER_API_KEY:
        raise ValueError("TOGETHER_API_KEY not set.")

    dataset_paths = {
        'hindi':  'data/bbc_hindi_articles_with_categories_cleaned.csv',
        'tamil':  'data/tamilmurasu_dataset.csv',
        'telugu': 'data/telugu_news_test.csv',
        'bangla': 'data/bangla_newspaper_dataset.csv'
    }

    # Models requested:
    models = [
        'qwen-3',            # via Together
        'gemini-2.5-pro',    # via Google GenAI
        'gemini-2.5-flash',  # via Google GenAI
        'gpt-4.5',           # via OpenAI
        'gpt-4o'   ,
        'gemma-3-12b-it',    
        'mistralai/Mistral-7B-Instruct-v0.2'
        
    ]

    router = LLMRouter(
        gemini_key=GEMINI_API_KEY,
        openai_key=OPENAI_API_KEY,
        together_key=TOGETHER_API_KEY
    )

    generator = NewsDatasetGenerator(
        llm_router=router,
        dataset_paths=dict(dataset_paths),
        models=models,
        output_dir='articles_today',
        output_csv='generated_dataset.csv',
        total_articles=8000,
        max_workers=50,
        # skip_seen_datapoints=True,
        shuffle_seed=97  # or an int for reproducible shuffling
    )
    generator.generate()        

[GENERATED] articles_today/hindi_gpt_4o_zero-shot_व_द_श_07231.txt
[GENERATED] articles_today/bangla_mistralai_Mistral_7B_Instruct_v0_2_chain-of-thought_technology_07232.txt
[GENERATED] articles_today/telugu_gpt_4o_zero-shot_eenadu_business_07233.txt
[GENERATED] articles_today/hindi_mistralai_Mistral_7B_Instruct_v0_2_zero-shot_व_द_श_07234.txt
[GENERATED] articles_today/tamil_mistralai_Mistral_7B_Instruct_v0_2_zero-shot_இந_த_ய_07235.txt
[GENERATED] articles_today/hindi_gpt_4o_chain-of-thought_मन_र_जन_07236.txt
[GENERATED] articles_today/telugu_mistralai_Mistral_7B_Instruct_v0_2_few-shot_eenadu_business_07237.txt
[GENERATED] articles_today/tamil_mistralai_Mistral_7B_Instruct_v0_2_chain-of-thought_வ_ள_ய_ட_ட_07238.txt
[GENERATED] articles_today/hindi_gpt_4o_zero-shot_भ_रत_07239.txt
[GENERATED] articles_today/hindi_gpt_4o_few-shot_ख_ल_07240.txt
[GENERATED] articles_today/hindi_mistralai_Mistral_7B_Instruct_v0_2_zero-shot_मन_र_जन_07241.txt
[GENERATED] articles_today/tamil_gpt_4o_zero-shot_வ_ள